# Reproducing Powell et al. (2021) from Zenodo record 3924718

**Paper.** Powell D, Haddad SA, Gorur-Shandilya S, Marder E (2021). Coupling between fast and slow
oscillator circuits in *Cancer borealis* is temperature-compensated. *eLife* 10:e60454.
[doi:10.7554/elife.60454](https://doi.org/10.7554/elife.60454). The claims file
(`claims/powell-2021.yaml`) was written from the version of record (Europe PMC, PMC7889077) with its
figure supplements, and from the paper's code, https://github.com/marderlab/gastric at commit
fa07d36a7e57fc7787a9a8e2fed72300b1a8e6e2.

**Dataset.** [ZENODO:3924718](https://zenodo.org/records/3924718), DOI 10.5281/zenodo.3924718,
record 3924718. The paper's data statement cites https://zenodo.org/record/3924718, this record,
and the deposit has no other version (concept 3924717 has the one version 3924718), so it is pinned.
Files read: `evoked_data.mat` whole (72,856,015 bytes, MD5 7b4062371133a07e373900cde8c42b15, checked
on loading), the zip directories of the eleven zip files (their central directories only, not
their contents), and the first 200 kB of the first ABF file of 941_003.zip and 941_006.zip and of
901_046_0006.abf in 901_046.zip, for their headers. Those partial reads have no MD5 to check.

**What the record holds.** `evoked_data.mat` holds the sorted PD, LG and DG spike times, the saline
temperature and a stimulation mask (1 kHz) of the ten evoked preparations of the paper's second set
(Methods, N = 10, 2017). The zip files hold the raw ABF recordings of nine of those ten (no zip for
901_005) and of two preparations recorded in June 2019 (941_003, 941_006) that are in no set the
paper describes, with no spike annotations. The spontaneous set (nine preparations of Haddad and
Marder 2018) and the set with the pyloric pacemaker hyperpolarized (nine preparations, 2020) are not
in the record. The record has no README; its description is the paper's abstract.

**Burst rule.** The Methods keep bursts of at least 5 spikes (LG, DG) or 2 (PD); the paper's code
(crabsort.computePeriods) compares the count with the number of ISIs in the burst, so it keeps
bursts of at least 6 or 3 spikes. Every claim is tested under the Methods rule, as the claims file
states it, and the code's rule is shown as a variant.

| Claim | Tested |
|---|---|
| fig2-spontaneous-11-to-30 | Blocked: data missing (spontaneous set not in the record) |
| fig2a-less-regular-at-high-temperature | Blocked: data missing (spontaneous set not in the record) |
| fig3-evoked-7-to-21-in-9-of-10 | Tested |
| fig3s1-spontaneous-and-evoked-similar | Blocked: data missing (no spike times of 941_006) |
| fig4b-periods-decrease-all-preps | Tested |
| fig4c-q10-about-2 | Tested |
| fig4c-q10-pd-lg-not-different | Tested |
| fig4s1-slowdown-after-stimulation | Tested |
| fig4s2-gastric-without-pyloric-pacemaker | Blocked: data missing (PD-hyperpolarized set not in the record) |
| fig4s2-rhythm-counts | Blocked: data missing (PD-hyperpolarized set not in the record) |
| fig5a-example-ratios | Tested |
| fig5d-lg-pd-integer-coupled | Tested |
| fig5d-dg-pd-not-coupled | Tested |
| fig5e-lg-area-exceeds-dg | Tested |
| fig5f-ratio-independent-of-temperature | Tested |
| fig5f-ratio-range | Tested |
| fig6b-lg-phase-locked | Tested |
| fig6b-lg-start-counts | Tested |
| fig6b-lg-phase-constant | Tested |
| fig6c-lg-spiking-drops-near-pd-end | Tested |
| fig6e-dg-not-phase-locked | Tested |
| fig6f-dg-spiking-no-drop | Tested |

In [ ]:
import time
import zipfile
import zlib
import re
import resource

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import stg

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
np.seterr(all="ignore")

started = time.time()
raw = stg.fetch()      # whole file into memory, MD5 checked against the record's list
preps = stg.load(raw)
del raw
names = [p["name"] for p in preps]
print(f"evoked_data.mat read and MD5 checked ({stg.FILE_MD5}); {len(preps)} preparations, "
      f"{time.time() - started:.0f} s")

# Bursts under the Methods rule (main) and the code's rule (variant), module stg.
B = stg.all_bursts(preps, "spikes")
BC = stg.all_bursts(preps, "isis")


def seed(claim, k=0):
    return zlib.crc32(f"powell-2021/{claim}".encode()) + k

## Data check

The record's file list, then what `evoked_data.mat` holds. Each check of the description against the
values is stated under its output.

In [ ]:
import requests
rec = requests.get(f"https://zenodo.org/api/records/{stg.RECORD}", timeout=60).json()
files = pd.DataFrame([{"key": f["key"], "size_GB": f["size"] / 1e9, "md5": f["checksum"]} for f in rec["files"]])
print("Record", rec["id"], "| version:", rec["metadata"].get("version"), "| published:", rec["metadata"]["publication_date"],
      "| resource type:", rec["metadata"]["resource_type"], "| access:", rec["metadata"]["access_right"])
print(files.sort_values("key").to_string(index=False))
print("Total size: %.1f GB" % files.size_GB.sum())

zip_rows = []
for key in sorted(files.key):
    if not key.endswith(".zip"):
        continue
    z, fh = stg.zip_listing(key)
    with fh:
        info = z.infolist()
        rawf = sorted(i.filename for i in info if i.filename.endswith((".abf", ".crab")))
        other = sorted(i.filename.rsplit("/", 1)[-1].rsplit(".", 1)[-1] for i in info
                       if not i.filename.endswith("/") and not i.filename.endswith((".abf", ".crab")))
        zip_rows.append({"zip": key, "abf": sum(f.endswith(".abf") for f in rawf), "crab": sum(f.endswith(".crab") for f in rawf),
                         "first": rawf[0].rsplit("/", 1)[-1], "last": rawf[-1].rsplit("/", 1)[-1],
                         "other_files": dict(pd.Series(other).value_counts()),
                         "first_file_date": str(z.getinfo(rawf[0]).date_time[:3])})
zips = pd.DataFrame(zip_rows)
print(zips.to_string(index=False))

The record has 12 files: `evoked_data.mat` and eleven zips of raw recordings. Zenodo types the
record as a journal article, not a dataset. The zips hold raw recordings, as ABF
files or as crabsort `.crab` files, and the crabsort network files (`network/<nerve>/*.mat`, the
trained spike classifiers). Whether the `.crab` files carry spike annotations was not examined; the
ABF-only zips (901_046, 901_098, 932_151, 941_003, 941_006) carry none. The number of raw files per
preparation differs from the number of files stacked in `evoked_data.mat` in some zips (printed
above and in the next table). There is no 901_005.zip, so one of the ten preparations has spike times but no
raw data. 941_003 and 941_006 are not among the ten preparations of `getEvokedData.m` and their files
date from June 2019 (the zip dates, and the ABF headers below), while the paper's evoked preparations
are from 2017 and the hyperpolarization set from 2020.

In [ ]:
import struct
# Recording date in the ABF header: ABF2 uFileStartDate (offset 16), ABF1 lFileStartDate (offset 20), YYYYMMDD.
for key, member in [("941_003.zip", "941_003/941_003_0000.abf"), ("941_006.zip", "941_006/941_006_0000.abf"),
                    ("901_046.zip", "901_046/901_046_0006.abf"), ("901_098.zip", "901_098/901_098_0000.abf"),
                    ("932_151.zip", "932_151/932_151_0000.abf")]:
    z, fh = stg.zip_listing(key)
    with fh:
        head = z.open(member).read(200_000)
    abf2 = head[:4] == b"ABF2"
    date = struct.unpack("<I", head[16:20])[0] if abf2 else struct.unpack("<i", head[20:24])[0]
    strings = [x.decode() for x in re.findall(rb"[ -~]{4,}", head) if len(x) < 40]
    print(member, "| format:", head[:4], "| recording date in header:", date,
          "| channel-like names:", sorted(set(x for x in strings if re.fullmatch(r"(IN \d+|Temp|Vm_\w+|Iout_\w+)", x))))

941_003 and 941_006 were recorded on 11 and 12 June 2019 (header dates), as ABF2 files with
intracellular channels (Vm_A1 to Vm_B2, Iout_A) and analog inputs named only IN 5 to IN 14 and Temp,
so the nerves are not named in the files. 901_046 and 901_098 were recorded in August and December
2017, as the Methods say of the evoked set. 932_151, one of the ten evoked preparations, was
recorded on 2 May 2019 by its header, while the Methods say the evoked preparations came from
crabs caught in 2017. The ABF1 files carry no channel names in the header read here.

In [ ]:
rows = []
for p, b, bc in zip(preps, B, BC):
    m = p["mask"]
    z = np.flatnonzero(~m)
    runs = np.split(z, np.flatnonzero(np.diff(z) > 1) + 1) if len(z) else []
    run_s = np.array([len(r) / 1e3 for r in runs])
    rows.append({
        "prep": p["name"], "objects": f'{p["experiment_object"]}/{p["filename_object"]}',
        "files": len(p["file_names"]), "first_file": p["file_names"][0], "last_file": p["file_names"][-1],
        "duration_s": len(p["temperature"]) / 1e3,
        "T_min": p["temperature"].min(), "T_max": p["temperature"].max(),
        "PD_spikes": len(p["PD"]), "LG_spikes": len(p["LG"]), "DG_spikes": len(p["DG"]),
        "last_spike_s": max(np.max(p[k]) if len(p[k]) else 0 for k in ("PD", "LG", "DG")),
        "PD_bursts": len(b["PD"]["starts"]), "LG_bursts": len(b["LG"]["starts"]), "DG_bursts": len(b["DG"]["starts"]),
        "LG_bursts_code": len(bc["LG"]["starts"]), "DG_bursts_code": len(bc["DG"]["starts"]),
        "mask0_runs": len(runs), "mask0_run_s_median": np.median(run_s) if len(runs) else np.nan,
        "mask0_frac": 1 - m.mean(), "empty_fields": ",".join(p["empty_fields"]),
    })
check = pd.DataFrame(rows)
print(check.round(2).to_string(index=False))
print("\nPreparation names, in order:", names)
print("getEvokedData.m lists (sorted):", stg.PREPS, "| same:", names == stg.PREPS)
print("Every 1 kHz sample has a file code:", all((p["file_codes"] > 0).all() for p in preps),
      "| file codes run in order:", all((np.diff(p["file_codes"].astype(int)) >= 0).all() for p in preps))
print("Samples per file (s):", sorted({round(np.bincount(p["file_codes"])[1:].mean() / 1e3, 1) for p in preps}))
print("Spike times sorted in the file:", all(True for _ in preps), "(sorted on loading, as getEvokedData.m does)")
print("Peak memory so far: %.1f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

The ten preparations are the ten of `getEvokedData.m`, in its sorted order, and each `filename`
category list runs from file 0000 without gaps, so every raw file of each preparation is in the
stacked data (the `.crab` zip of 901_062 holds 101 files against 58 stacked; the others match).
Spike times run to the end of each recording. Temperatures range from about 6 to
23 C, as the Methods describe (steps from 7 C, typically to 21 or 23 C). 901_005 has no DG spikes
at all (its DG field is empty), so DG analyses have nine preparations. The fields `T` and
`time_offset` are empty in every preparation. The mask is 0 in runs of about 7 s, which fits the
stimulus trains the Methods describe and the code's `findStimulationTimes` (samples where a channel
exceeds 9, the stimulus artefact); the record does not say what the mask is.

In [ ]:
held = pd.DataFrame({p["name"]: stg.held_seconds(p) for p in preps}).T
print("Seconds of recording in each temperature bin (centre +- 1 C):")
print(held.round(0).astype(int).to_string())

Most preparations were held at 7, 11, 15, 19 and 21 C (4 C steps, each held 8 to 90 minutes) and
pass through 9, 13 and 17 C only on the way, for 20 to 270 s; 901_005 and 901_046 were stepped by
2 C, as the Methods allow, and only they reach 23 C. 901_062 never reaches 21 C.

In [ ]:
# Clocks. (1) Stimulation mask against LG spikes: LG bursting should follow each stimulation.
# LG burst starts in the 300 s after the end of each stimulation block (mask 0, blocks separated
# by > 120 s) against the 300 s before its first train, and with the mask shifted by +-1000 s.
def lg_after_before(p, b, shift_s=0.0):
    stim = np.flatnonzero(~p["mask"]) / 1e3 + shift_s
    if len(stim) == 0:
        return np.nan, np.nan
    gaps = np.flatnonzero(np.diff(stim) > 120)
    firsts = np.r_[stim[0], stim[gaps + 1]]
    lasts = np.r_[stim[gaps], stim[-1]]
    s = b["LG"]["starts"]
    after = np.mean([np.sum((s > l) & (s < l + 300)) for l in lasts])
    before = np.mean([np.sum((s > f - 300) & (s < f)) for f in firsts])
    return after, before

rows = []
for p, b in zip(preps, B):
    r = {"prep": p["name"]}
    for sh in (0, 1000, -1000):
        a, bf = lg_after_before(p, b, sh)
        r[f"after_{sh}"], r[f"before_{sh}"] = a, bf
    rows.append(r)
clock = pd.DataFrame(rows)
print("Mean LG bursts in 300 s after the end / before the start of a stimulation block, mask shifted by 0, +1000, -1000 s:")
print(clock.round(1).to_string(index=False))

# (2) Temperature against PD spikes: Spearman correlation of PD burst frequency with temperature at
# the burst start, with the temperature trace as recorded and shifted by 1000 s.
rows = []
for p, b in zip(preps, B):
    per = b["PD"]["periods"]
    ok = np.isfinite(per) & (per < 5)
    st = b["PD"]["starts"][ok]
    r0 = stats.spearmanr(stg.temp_at(p, st), 1 / per[ok])[0]
    sh = np.clip(st + 1000, 0, len(p["temperature"]) / 1e3 - 0.001)
    r1 = stats.spearmanr(stg.temp_at(p, sh), 1 / per[ok])[0]
    rows.append({"prep": p["name"], "rho_aligned": r0, "rho_shift_1000s": r1})
print("\nPD burst frequency against temperature (Spearman rho):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

In every preparation LG bursts more in the 300 s after a stimulation block than in the 300 s
before it (16 to 40 bursts against 0.6 to 21), and the contrast shrinks or reverses when the mask is
shifted by 1000 s in either direction (printed above); the mask and the LG spike times share a
clock. PD burst frequency follows the temperature trace (rho 0.83 to 0.96) and much less closely
when the trace is shifted by 1000 s (0.04 to 0.37, and 0.78 in 901_046, whose temperature changes
slowly), so the temperature and the PD spike times share a clock. DG and LG spike times come from the same stacked recordings as PD (one
`crabsort.consolidate` call per preparation in `getEvokedData.m`); their clock is consistent with
the data (DG and LG bursts both follow stimulation, see the LG check) but not separately shown.

In [ ]:
# Recordings deposited twice: shared spike times between preparations (PD, to the millisecond),
# and identical durations or spike counts.
dup = []
keys = [np.unique(np.round(p["PD"] * 1e3).astype(np.int64)) for p in preps]
for i in range(len(preps)):
    for j in range(i + 1, len(preps)):
        shared = len(np.intersect1d(keys[i], keys[j])) / min(len(keys[i]), len(keys[j]))
        same_len = len(preps[i]["temperature"]) == len(preps[j]["temperature"])
        same_counts = all(len(preps[i][k]) == len(preps[j][k]) for k in ("PD", "LG"))
        if shared > 0.5 or same_len or same_counts:
            dup.append((names[i], names[j], round(shared, 3), same_len, same_counts))
print("Pairs with > 50% shared PD spike times, equal duration or equal PD and LG counts:", dup if dup else "none")
print("Largest fraction of shared PD spike times (ms) between two preparations: %.3f" % max(
    len(np.intersect1d(keys[i], keys[j])) / min(len(keys[i]), len(keys[j]))
    for i in range(len(preps)) for j in range(i + 1, len(preps))))

No two preparations share their spike trains, durations or spike counts beyond chance coincidences
at 1 ms, so no recording is deposited twice in `evoked_data.mat`.

## fig3-evoked-7-to-21-in-9-of-10

> Gastric mill rhythms could be evoked by dpon stimulation across 7 to 21 C in 9 of 10
> preparations.

A rhythm is present in a bin under reading (a) when the preparation has at least one LG burst
period there, under (b) when it has at least 30 LG significands (Figure 5's ratios). "Every bin the
preparation was held at": the claims file leaves the meaning of "held" to the author; a bin is
taken as held when the recording spends at least 120 s in it, the minimum duration of a step in
the Methods ("a given temperature step was held for at least 2 minutes" for the spontaneous set;
`BurstPeriodsVsTime.m` uses the same 120 s for the evoked set). Counts under any held bin
(more than 0 s) are shown too.

In [ ]:
def rhythm_table(Bx):
    rows = []
    for p, b in zip(preps, Bx):
        lgT = stg.temp_at(p, b["LG"]["starts"])
        ic = stg.integer_coupling(p, b, "LG")
        h = stg.held_seconds(p)
        for c in stg.BIN_CENTRES:
            rows.append({"prep": p["name"], "bin": int(c), "held_s": h[int(c)],
                         "lg_periods": int(np.sum(stg.in_bin(lgT, c) & np.isfinite(b["LG"]["periods"]))),
                         "significands": int(np.sum(stg.in_bin(ic["temp"], c) & np.isfinite(ic["rem"])))})
    return pd.DataFrame(rows)


def evoked_7_21(tab, min_held, reading):
    out = {}
    for prep, g in tab.groupby("prep", sort=False):
        g = g[(g.bin >= 7) & (g.bin <= 21)]
        held_bins = g[g.held_s >= min_held] if min_held > 0 else g[g.held_s > 0]
        present = held_bins.lg_periods >= 1 if reading == "a" else held_bins.significands >= 30
        has_7_and_21 = set(held_bins.bin[present]) >= {7, 21}
        out[prep] = bool(has_7_and_21 and present.all())
    return out

for label, Bx in (("Methods rule", B), ("code rule", BC)):
    tab = rhythm_table(Bx)
    if label == "Methods rule":
        tab_main = tab
    print(f"--- {label}: LG burst periods per bin")
    print(tab.pivot(index="prep", columns="bin", values="lg_periods").to_string())
    print("significands per bin")
    print(tab.pivot(index="prep", columns="bin", values="significands").to_string())
    for min_held in (120, 0):
        for reading in ("a", "b"):
            r = evoked_7_21(tab, min_held, reading)
            print(f"held >= {min_held} s, reading ({reading}): {sum(r.values())} of 10 with a rhythm at every held bin "
                  f"from 7 to 21 C; without: {[k for k, v in r.items() if not v]}")

In [ ]:
# The two preparations without a rhythm at 21 C: LG spikes and seconds in the 21 C bin, and the
# warmest bin with an LG burst.
for name in ("901_062", "901_086"):
    p = preps[names.index(name)]
    in21 = stg.in_bin(p["temperature"], 21)
    lg_t = stg.temp_at(p, p["LG"])
    b = B[names.index(name)]
    lgT = stg.temp_at(p, b["LG"]["starts"])
    print(name, "| seconds in 21 C bin: %.0f" % (in21.sum() / 1e3), "| LG spikes there:", int(np.sum(stg.in_bin(lg_t, 21))),
          "| stimulation samples there:", int(np.sum(in21 & ~p["mask"])),
          "| warmest LG burst start at %.1f C" % np.nanmax(lgT))

**Result: not reproduced.** Under reading (a), with either burst rule, 8 of the 10 preparations
have an evoked LG rhythm at every held bin from 7 to 21 C, not 9; under reading (b) (at least 30
significands) only 2 of 10 do. 901_062 never reaches 21 C, and
901_086 was held at 21 C for 330 s, was stimulated there, and has no LG burst there; its warmest LG
burst is near 19 C. Counting every bin the temperature passed through, not only held ones, gives
fewer (3 and 1 of 10). The paper's "9 of 10" is one more than the data hold.

## fig4b-periods-decrease-all-preps

> Both LG (gastric mill) and PD (pyloric) burst periods decrease with increasing temperature, a
> coordinated decrease seen in all 10 preparations.

Per preparation and bin, the mean period of bursts starting in the bin (PD periods above 5 s and LG
above 50 s removed, `fig_burst_period.m`). Reading (a): the warmest bin's mean is below the
coldest's; (b): Spearman rho of bin mean against bin temperature is negative.

In [ ]:
def period_means(p, b, neuron, limit):
    per = b[neuron]["periods"].copy()
    per[per > limit] = np.nan
    T = stg.temp_at(p, b[neuron]["starts"])
    out = {}
    for c in stg.BIN_CENTRES:
        v = per[stg.in_bin(T, c) & np.isfinite(per)]
        if len(v):
            out[int(c)] = float(np.mean(v))
    return out

for label, Bx in (("Methods rule", B), ("code rule", BC)):
    rows = []
    for p, b in zip(preps, Bx):
        r = {"prep": p["name"]}
        for neuron, limit in (("PD", 5), ("LG", 50)):
            m = period_means(p, b, neuron, limit)
            cs = sorted(m)
            r[f"{neuron}_coldest"] = f"{cs[0]}C {m[cs[0]]:.2f}"
            r[f"{neuron}_warmest"] = f"{cs[-1]}C {m[cs[-1]]:.2f}"
            r[f"{neuron}_a"] = m[cs[-1]] < m[cs[0]]
            r[f"{neuron}_rho"] = stats.spearmanr(cs, [m[c] for c in cs])[0]
            r[f"{neuron}_b"] = r[f"{neuron}_rho"] < 0
        rows.append(r)
    dec = pd.DataFrame(rows)
    print(f"--- {label} (bin means of burst period, s)")
    print(dec.round(3).to_string(index=False))
    for rd in ("a", "b"):
        print(f"reading ({rd}): {int((dec[f'PD_{rd}'] & dec[f'LG_{rd}']).sum())} of 10 decrease for both PD and LG")
    if label == "Methods rule":
        dec_main = dec

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for k, (neuron, limit) in enumerate((("PD", 5), ("LG", 50))):
    allv = []
    for p, b in zip(preps, B):
        m = period_means(p, b, neuron, limit)
        ax[k].plot(list(m), list(m.values()), "o-", alpha=.6)
        allv += list(m.values())
    ax[k].set_yscale("log"); ax[k].set_xlabel("Temperature bin (C)"); ax[k].set_ylabel(f"{neuron} mean burst period (s)")
    print(f"{neuron} bin means range {min(allv):.3f} to {max(allv):.3f} s, all plotted")
plt.tight_layout(); plt.show()

**Result: reproduced.** In all 10 preparations the mean PD and LG burst periods in the warmest bin
are below those in the coldest bin, and the Spearman correlation of bin mean with temperature is
negative for both (PD rho -0.98 to -1, LG rho -0.64 to -1), under both burst rules.

## fig4c-q10-about-2

> Most preparations (8 of 10) had a cycle period Q10 of about 2 for both the pyloric and the
> gastric mill rhythm.

The Methods' Q10 (one per temperature, from the mean frequency at T against 11 C, averaged per
preparation) is the main analysis; the code's per-burst Q10 is a variant. Which temperatures enter
the Methods' Q10 is not fixed by the claims file. The Methods compute it "at every temperature",
the temperatures being the steps of the protocol, held "for the duration of evoked gastric mill
activity" (Methods, "Spontaneous and evoked gastric mill rhythm temperature ranges"). Most
preparations pass through 9, 13 and 17 C only on the way between 4 C steps (data check), with 1 to
17 LG bursts there; a Q10 from such a bin next to 11 C is raised to the power 5. Two readings are
run: (i) every bin with bursts, (ii) only bins held at least 120 s (the temperature steps).

In [ ]:
def q10_table(Bx):
    rows = []
    for p, b in zip(preps, Bx):
        steps = [c for c, s in stg.held_seconds(p).items() if s >= 120]
        r = {"prep": p["name"]}
        for neuron, limit in (("PD", 5), ("LG", 50)):
            r[f"{neuron}_all_bins"], q_all = stg.q10_methods(p, b, neuron, limit)
            r[f"{neuron}_steps"], _ = stg.q10_methods(p, b, neuron, limit, bins=steps)
            r[f"{neuron}_code"] = stg.q10_code(p, b, neuron, limit)
            r[f"{neuron}_per_bin"] = {c: round(v, 2) for c, v in q_all.items()}
        rows.append(r)
    return pd.DataFrame(rows)

ranges = {"a (1.8-2.2)": (1.8, 2.2), "b (1.5-2.5)": (1.5, 2.5)}
q10s = {}
for label, Bx in (("Methods rule", B), ("code rule", BC)):
    q = q10_table(Bx)
    q10s[label] = q
    print(f"--- {label}")
    print(q[["prep", "PD_all_bins", "LG_all_bins", "PD_steps", "LG_steps", "PD_code", "LG_code"]].round(3).to_string(index=False))
    for version in ("all_bins", "steps", "code"):
        for rname, (lo, hi) in ranges.items():
            n = int((q[f"PD_{version}"].between(lo, hi) & q[f"LG_{version}"].between(lo, hi)).sum())
            print(f"  Q10 {version:9s} range {rname}: {n} of 10 with both PD and LG in range")
print("\nPer-bin LG Q10s, Methods rule, every bin (the outliers come from the 9, 13 and 17 C bins):")
print(q10s["Methods rule"][["prep", "LG_per_bin"]].to_string(index=False))

**Result: not reproduced.** Under the Methods' Q10, the main analysis, 1 of 10 (range 1.8 to 2.2)
or 2 of 10 (1.5 to 2.5) preparations have both Q10s near 2 when every bin is used, and 1 or 3 of 10
when only the held steps are used (Methods burst rule; 1 and 3 with the code's); none reaches 8. LG Q10s from every bin reach
16.6 and 39.0 in two preparations, from bins next to 11 C crossed during a ramp. The code's per-burst
Q10, a variant, gives 8 of 10 in the wider range (1.5 to 2.5) under both burst rules and 3 (Methods
rule) or 1 (code rule) of 10 in the narrower one, so the paper's count matches the code's
computation under the wider reading of "~2", not the Methods' formula.

## fig4c-q10-pd-lg-not-different

> The apparent Q10s of the pyloric (PD) and gastric mill (LG) burst frequencies do not differ
> (paired t-test, p = 0.235, N = 10, t = 1.27).

A claim of absence: two-sided paired t-test of the per-preparation Methods Q10s, reproduced when
p >= 0.05. The code's per-burst Q10 and the held-steps reading are shown as variants.

In [ ]:
for label in ("Methods rule", "code rule"):
    q = q10s[label]
    for version in ("all_bins", "steps", "code"):
        ok = q[f"PD_{version}"].notna() & q[f"LG_{version}"].notna()
        t, pval = stats.ttest_rel(q[f"PD_{version}"][ok], q[f"LG_{version}"][ok])
        d = q[f"PD_{version}"][ok] - q[f"LG_{version}"][ok]
        print(f"{label:12s} Q10 {version:9s}: n = {ok.sum()}, mean PD - LG = {d.mean():+.3f}, t = {t:.3f}, p = {pval:.3f}")

**Result: reproduced.** The paired t-test on the Methods' Q10s (every bin) finds no difference
(t = -1.42, p = 0.19, n = 10, Methods burst rule; printed above). The code's per-burst Q10 with the
code's burst rule gives t = 1.272, p = 0.235, the paper's values exactly; with the Methods burst rule
t = 1.20, p = 0.26. No version tested reaches p < 0.05 (printed above).

## fig4s1-slowdown-after-stimulation

> The evoked gastric mill rhythm slows down (LG burst period lengthens) with time since
> stimulation.

As `gastric.BurstPeriodsVsTime` and `fig_evoked_slowdown.m`: stretches after each stimulation held
at one rounded temperature for at least 120 s, LG periods above 30 s removed, time from the first
LG burst, each stretch interpolated at 1 s over 0 to 600 s, mean over stretches per temperature,
least-squares line of the mean on time. LG bursts with the code's rule of this script (7 ISIs) and
the Methods' (5 spikes).

In [ ]:
tt = np.arange(0, 601)
slow_res = {}
for label, kw in (("code, 7 ISIs", dict(min_n=7, count="isis")), ("Methods, 5 spikes", dict(min_n=5, count="spikes"))):
    rows, temps, per_prep = [], [], []
    for p in preps:
        lg = stg.bursts(p["LG"], stg.IBI["LG"], **kw)
        for s in stg.stretches_after_stimulation(p, lg):
            ok = np.isfinite(s["periods"])
            if ok.sum() >= 2:
                rows.append(np.interp(tt, s["time"][ok], s["periods"][ok], left=np.nan, right=np.nan))
            else:
                rows.append(np.full(len(tt), np.nan))
            temps.append(s["temperature"])
            per_prep.append(p["name"])
    A, temps = np.array(rows), np.array(temps)
    fits = []
    for c in stg.BIN_CENTRES:
        sel = temps == c
        if not sel.any():
            continue
        M = np.nanmean(A[sel], 0)
        ok = np.isfinite(M)
        if ok.sum() < 2:
            continue
        slope, icpt = np.polyfit(tt[ok], M[ok], 1)
        r = np.corrcoef(M[ok], slope * tt[ok] + icpt)[0, 1]
        fits.append({"T": int(c), "stretches": int(sel.sum()), "preps": len(set(np.array(per_prep)[sel])),
                     "seconds_with_data": int(ok.sum()), "mean_period_0_60s": np.nanmean(M[:61]),
                     "mean_period_540_600s": np.nanmean(M[540:]), "slope_s_per_s": slope, "r": r})
    fits = pd.DataFrame(fits)
    slow_res[label] = fits
    print(f"--- LG bursts: {label}; {len(temps)} stretches from {len(set(per_prep))} preparations")
    print(fits.round(4).to_string(index=False))
    print(f"positive slope at {int((fits.slope_s_per_s > 0).sum())} of {len(fits)} temperatures")

**Result: reproduced.** The fitted slope of mean LG period on time since stimulation is positive at
8 of the 9 temperatures with data under both burst rules, all but 13 C (slope -0.0006 s per s, two
stretches). At the five temperatures with 8 or more stretches (7, 11, 15, 19 and 21 C) the slopes
are 0.0055 to 0.0171 s per s with r from 0.63 to 0.87, and the mean period rises from 6.7 to 15.4 s
in the first minute to 10.1 to 19.6 s at 9 to 10 minutes. The 9, 13, 17 and 23 C means rest on one
or two stretches.

## fig5a-example-ratios

> In the example preparation the ratio of LG to PD period is close to an integer at both
> temperatures, about 10 at 7 C and about 11 at 23 C, and PD bursts end shortly before LG bursts
> start.

Preparation 901_046, files 0006 and 0079 (`fig_integer_coupling.m`), selected through the per-sample
`filename` codes. Each LG cycle starting in the file, with its mean PD period
(`gastric.integerCoupling`). Reading (a): the cycle whose LG period is closest to the labelled
T_LG; (b): the median ratio of the file.

In [ ]:
p, b = preps[names.index("901_046")], B[names.index("901_046")]
ic = stg.integer_coupling(p, b, "LG")
ex = {}
for fname, T_LG, label in (("901_046_0006", 9.17, 10.06), ("901_046_0079", 3.44, 10.98)):
    codeval = p["file_names"].index(fname) + 1
    idx = np.flatnonzero(p["file_codes"] == codeval)
    t0, t1 = idx[0] / 1e3, (idx[-1] + 1) / 1e3
    u = (b["LG"]["starts"] >= t0) & (b["LG"]["starts"] < t1)
    starts = b["LG"]["starts"][u]
    pd_end_before = [t - b["PD"]["ends"][b["PD"]["ends"] < t].max() for t in starts]
    df = pd.DataFrame({"LG_start_s": starts - t0, "T_LG": ic["y"][u], "mean_T_PD": ic["x"][u], "ratio": ic["ratio"][u],
                       "PD_end_to_LG_start_s": pd_end_before})
    print(f"{fname}: {t0:.0f}-{t1:.0f} s, mean temperature {p['temperature'][idx].mean():.2f} C, {u.sum()} LG bursts")
    print(df.round(3).to_string(index=False))
    j = np.nanargmin(np.abs(df.T_LG.values - T_LG))
    near = df[np.abs(df.T_LG - df.T_LG.iloc[j]) < 0.01]
    ra, rb = df.ratio.iloc[j], np.nanmedian(df.ratio)
    ex[fname] = (ra, rb)
    for name, r in (("(a) closest cycle", ra), ("(b) median", rb)):
        print(f"  {name}: ratio {r:.3f}; label {label}, within 10%: {abs(r - label) <= 0.1 * label}; "
              f"distance to integer {abs(r - round(r)):.3f}, within 0.2: {abs(r - round(r)) <= 0.2}")
    if len(near) > 1:
        print("  cycles within 0.01 s of the closest T_LG:", near[["T_LG", "mean_T_PD", "ratio"]].round(3).values.tolist())
    print(f"  PD burst end to LG burst start: median {np.median(pd_end_before):.3f} s, range {np.min(pd_end_before):.3f} to {np.max(pd_end_before):.3f} s")

**Result: reproduced.** Under reading (a), the LG cycle closest to the labelled T_LG gives a ratio
of 10.96 at 7 C (label 10.06, distance to an integer 0.04) and 10.87 at 23 C (label 10.98, distance
0.13), both within 10% of the labels and within 0.2 of an integer. Another 7 C cycle with nearly the
same T_LG (9.245 against 9.244 s) has ratio 10.02 and mean PD period 0.923 s, close to the label's
0.911 s. Under reading (b) the median ratios are 10.76 and 14.02, which miss. LG bursts start a
median 0.17 s (7 C) and 0.13 s (23 C) after the last PD burst end (ranges 0.09 to 0.32 s and 0.001
to 0.32 s); this part is shown, not judged.

## fig5d-lg-pd-integer-coupled

> LG and PD burst periods are significantly integer-coupled at every temperature from 7 to 21 C
> (two-sample Kolmogorov-Smirnov test against shuffled periods, p < 1e-5, D > 0.24).

Significands pooled over preparations per bin, against shuffled significands: (a) the code's 100
independent draws of LG and mean PD periods; (b) a permutation of the bin's mean PD periods. Each
repeated 1000 times; median D and p (asymptotic p, as MATLAB's `kstest2`). Seeds from the claim id,
and four more (+1000 to +4000).

In [ ]:
def pooled_coupling(Bx, slow):
    ics = [stg.integer_coupling(p, b, slow) for p, b in zip(preps, Bx)]
    cat = {k: np.concatenate([i[k] for i in ics]) for k in ("x", "y", "temp", "rem", "ratio")}
    cat["prep"] = np.concatenate([np.full(len(i["y"]), k) for k, i in enumerate(ics)])
    return cat


def ks_by_bin(cat, claim, reading, k_seed=0, n_rep=1000):
    rng = np.random.default_rng(seed(claim, k_seed))
    rows = []
    for c in stg.BIN_CENTRES:
        u = stg.in_bin(cat["temp"], c)
        obs = cat["rem"][u]
        obs = obs[np.isfinite(obs)]
        ok = u & np.isfinite(cat["x"]) & np.isfinite(cat["y"])
        if len(obs) < 2:
            continue
        res = stg.ks_shuffled(obs, cat["x"][ok], cat["y"][ok], rng, reading, n_rep)
        rows.append({"T_C": int(c), "n": len(obs), "D": np.median(res[:, 0]), "p": np.median(res[:, 1]),
                     "D_2.5pct": np.percentile(res[:, 0], 2.5), "p_97.5pct": np.percentile(res[:, 1], 97.5)})
    return pd.DataFrame(rows)


ALPHA = 0.05 / 9
claim = "fig5d-lg-pd-integer-coupled"
cat_lg = {"Methods rule": pooled_coupling(B, "LG"), "code rule": pooled_coupling(BC, "LG")}
ks_lg = {}
for label, cat in cat_lg.items():
    for reading in ("draw", "perm"):
        df = ks_by_bin(cat, claim, reading)
        ks_lg[(label, reading)] = df
        sel = df[(df.T_C >= 7) & (df.T_C <= 21)]
        print(f"--- {label}, shuffle {reading}")
        print(df.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
        print(f"7-21 C: all p < 0.05/9: {bool((sel.p < ALPHA).all())}; all D > 0.216: {bool((sel.D > 0.216).all())}; "
              f"all p < 1e-5: {bool((sel.p < 1e-5).all())}; smallest D {sel.D.min():.3f} at {int(sel.T_C[sel.D.idxmin()])} C; largest p {sel.p.max():.2g}")

In [ ]:
# Seeds: the claim's own and +1000 to +4000, main reading (Methods rule, the code's draws).
rows = []
for k in (0, 1000, 2000, 3000, 4000):
    df = ks_by_bin(cat_lg["Methods rule"], claim, "draw", k)
    sel = df[(df.T_C >= 7) & (df.T_C <= 21)]
    rows.append({"seed_offset": k, "D_21C": float(sel.D[sel.T_C == 21].iloc[0]), "min_D_7_21": sel.D.min(),
                 "max_p_7_21": sel.p.max(), "holds": bool((sel.p < ALPHA).all() and (sel.D > 0.216).all())})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** With the code's shuffle (reading a), every bin from 7 to 21 C is significant
at 0.05/9, with median D from 0.223 (21 C) to 0.351 and median p from 1.5e-8 to 4.9e-4 (Methods
burst rule); D at 21 C is 0.221 to 0.224 under the five seeds, above the 0.216 bound in each (printed
above). The reported p < 1e-5 is not met at 17, 19 and 21 C with the code's 100 draws; with the
permutation shuffle (reading b) p is below 1e-5 at every bin from 7 to 21 C but D at 21 C is 0.199,
under the bound. The 23 C bin is not significant (D 0.162, p 0.31), as the paper's range 7 to 21 C
implies.

## fig5d-dg-pd-not-coupled

> DG and PD burst periods are not significantly integer-coupled at any temperature (p > 0.02,
> D < 0.16).

A claim of absence, the same test with DG (the code's shuffle, median over repetitions), at
0.05/9. The permutation shuffle is a variant.

In [ ]:
claim = "fig5d-dg-pd-not-coupled"
cat_dg = {"Methods rule": pooled_coupling(B, "DG"), "code rule": pooled_coupling(BC, "DG")}
for label, cat in cat_dg.items():
    for reading in ("draw", "perm"):
        df = ks_by_bin(cat, claim, reading)
        print(f"--- {label}, shuffle {reading}")
        print(df.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
        print(f"bins with p < 0.05/9: {df.T_C[df.p < ALPHA].tolist()}; D range {df.D.min():.3f}-{df.D.max():.3f}; p range {df.p.min():.2g}-{df.p.max():.2g}")
rows = []
for k in (0, 1000, 2000, 3000, 4000):
    df = ks_by_bin(cat_dg["Methods rule"], claim, "draw", k)
    rows.append({"seed_offset": k, "bins_significant": df.T_C[df.p < ALPHA].tolist(), "min_p": df.p.min(), "max_D": df.D.max()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** With the code's shuffle no temperature bin reaches p < 0.05/9 (median p 0.082
to 0.925, D 0.115 to 0.160, Methods burst rule), under all five seeds (smallest p 0.078 to 0.084).
p > 0.02 holds everywhere and D < 0.16 everywhere but the 23 C bin (12 DG cycles, D 0.160, or 0.19
with the code's burst rule). With the permutation shuffle (variant) the 11 and 15 C bins are
significant (p 5e-4 and 6e-4), so the absence depends on the shuffle.

## fig5e-lg-area-exceeds-dg

> The area between the significand c.d.f. and the diagonal is larger for LG/PD than for DG/PD at
> every temperature, and the DG/PD areas overlap those of shuffled periods.

As `plotIntegerCoupling.m`: per preparation and bin with at least 30 significands, the mean over 100
bootstrap resamples of the area; the shuffled area from as many significands drawn from one
shuffle of all gastric against all mean PD periods; mean and SEM over preparations. A bin with one
preparation has no SEM; its interval is its value.

In [ ]:
def areas(cat, rng):
    ok = np.isfinite(cat["x"]) & np.isfinite(cat["y"])
    rand_rem = np.mod(cat["y"] / cat["x"][rng.permutation(len(cat["x"]))], 1)
    rand_rem = rand_rem[np.isfinite(rand_rem)]
    A = np.full((len(stg.BIN_CENTRES), 10), np.nan)
    R = A.copy()
    for i, c in enumerate(stg.BIN_CENTRES):
        for j in range(10):
            v = cat["rem"][stg.in_bin(cat["temp"], c) & (cat["prep"] == j)]
            v = v[np.isfinite(v)]
            if len(v) < 30:
                continue
            rr = rand_rem[rng.integers(0, len(rand_rem), len(v))]
            A[i, j] = stg.integerness(v, rng)
            R[i, j] = stg.integerness(rr, rng)
    def ms(X):
        n = np.sum(np.isfinite(X), 1)
        sem = np.where(n > 1, np.nanstd(X, 1, ddof=1) / np.sqrt(n), 0.0)
        return np.nanmean(X, 1), sem, n
    return ms(A), ms(R)


def area_test(k_seed=0, Bcat=None):
    rng = np.random.default_rng(seed("fig5e-lg-area-exceeds-dg", k_seed))
    Bcat = Bcat or (cat_lg["Methods rule"], cat_dg["Methods rule"])
    (lm, ls, ln), (lrm, lrs, _) = areas(Bcat[0], rng)
    (dm, dsem, dn), (drm, drs, _) = areas(Bcat[1], rng)
    t = pd.DataFrame({"T": stg.BIN_CENTRES, "LG_n": ln, "LG_area": lm, "LG_sem": ls, "DG_n": dn, "DG_area": dm, "DG_sem": dsem,
                      "shufDG_area": drm, "shufDG_sem": drs, "shufLG_area": lrm, "shufLG_sem": lrs})
    both = t.LG_area.notna() & t.DG_area.notna()
    t["LG_gt_DG"] = np.where(both, t.LG_area > t.DG_area, None)
    t["DG_overlaps_shufDG"] = np.where(t.DG_area.notna(),
        (t.DG_area - t.DG_sem <= t.shufDG_area + t.shufDG_sem) & (t.DG_area + t.DG_sem >= t.shufDG_area - t.shufDG_sem), None)
    t["DG_overlaps_shufLG"] = np.where(t.DG_area.notna(),
        (t.DG_area - t.DG_sem <= t.shufLG_area + t.shufLG_sem) & (t.DG_area + t.DG_sem >= t.shufLG_area - t.shufLG_sem), None)
    return t

t = area_test()
print(t.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
dgb = t[t.DG_area.notna()]
print(f"LG > DG in {int((t.LG_gt_DG == True).sum())} of {int(t.LG_gt_DG.notna().sum())} bins with both; "
      f"DG overlaps its shuffle in {int((dgb.DG_overlaps_shufDG == True).sum())} of {len(dgb)} bins "
      f"(LG's shuffle: {int((dgb.DG_overlaps_shufLG == True).sum())})")
rows = []
for k in (0, 1000, 2000, 3000, 4000):
    tk = area_test(k)
    d = tk[tk.DG_area.notna()]
    rows.append({"seed_offset": k, "LG_gt_DG_all": bool((tk.LG_gt_DG.dropna() == True).all()),
                 "DG_overlap_bins": int((d.DG_overlaps_shufDG == True).sum()), "of": len(d)})
print(pd.DataFrame(rows).to_string(index=False))
tc = area_test(0, (cat_lg["code rule"], cat_dg["code rule"]))
dc = tc[tc.DG_area.notna()]
print(f"code burst rule: LG > DG in all bins with both: {bool((tc.LG_gt_DG.dropna() == True).all())}; "
      f"DG overlaps its shuffle in {int((dc.DG_overlaps_shufDG == True).sum())} of {len(dc)} bins")

**Result: partially reproduced.** The LG area exceeds the DG area in every bin where both exist
(8 of 8, 0.127 to 0.193 against 0.050 to 0.079), under all five seeds. The DG area (mean plus or
minus SEM) overlaps the shuffled area in 1 of 8 bins under the claim's seed and 1 to 4 of 8 under
the five seeds, never in most; in every bin but 7 C the DG area lies above the shuffled one. Against the LG call's shuffle, the other gray line of the figure, DG overlaps in 3 of 8.

## fig5f-ratio-independent-of-temperature

> The ratio of LG to PD cycle period does not change with temperature (Spearman test, p = 0.88,
> n = 3799).

A claim of absence: Spearman correlation of every LG/mean PD ratio with the temperature at the LG
burst start, pooled over preparations; reproduced when p >= 0.05. The code's prep-by-prep means per
bin are a variant.

In [ ]:
for label, cat in cat_lg.items():
    ok = np.isfinite(cat["ratio"])
    rho, pval = stats.spearmanr(cat["temp"][ok], cat["ratio"][ok])
    print(f"{label}: n = {ok.sum()} cycles, rho = {rho:.4f}, p = {pval:.3f}")
    rows = []
    for c in stg.BIN_CENTRES:
        for j in range(10):
            v = cat["ratio"][ok & (cat["prep"] == j) & stg.in_bin(cat["temp"], c)]
            if len(v):
                rows.append((c, v.mean()))
    rows = np.array(rows)
    r2, p2 = stats.spearmanr(rows[:, 0], rows[:, 1])
    print(f"   prep-by-prep means per bin: n = {len(rows)}, rho = {r2:.3f}, p = {p2:.3f}")
cat = cat_lg["Methods rule"]
ok = np.isfinite(cat["ratio"])
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(cat["temp"][ok], cat["ratio"][ok], ".", ms=2)
ax.set_ylim(0, 30); ax.set_xlabel("Temperature (C)"); ax.set_ylabel("T_LG / mean T_PD")
print("ratios above 30, outside the axes:", int(np.sum(cat["ratio"][ok] > 30)), "of", int(ok.sum()))
plt.show()

**Result: reproduced.** Pooled over all cycles the ratio does not correlate with temperature: rho =
0.016, p = 0.31, n = 3850 (Methods burst rule). With the code's burst rule n = 3799 and p = 0.880,
the paper's numbers exactly. The prep-by-prep means per bin do not correlate either (rho 0.21,
p = 0.10, 64 means).

## fig5f-ratio-range

> At low temperatures the ratios of LG to PD cycle period range widely, from about 5 to 25.

Reading (a) the 7 C bin, (b) bins 7 to 11 C. Reproduced when the 2.5th to 97.5th percentiles lie
within 4.5 to 27.5 and span at least 5.5 to 22.5.

In [ ]:
for label, cat in cat_lg.items():
    ok = np.isfinite(cat["ratio"])
    for rd, bins in (("a", (7,)), ("b", (7, 9, 11))):
        u = ok & np.any([stg.in_bin(cat["temp"], c) for c in bins], 0)
        v = cat["ratio"][u]
        lo, hi = np.percentile(v, [2.5, 97.5])
        inside = lo >= 4.5 and hi <= 27.5
        spans = lo <= 5.5 and hi >= 22.5
        print(f"{label}, reading ({rd}) bins {bins}: n = {u.sum()}, min {v.min():.2f}, 2.5% {lo:.2f}, 97.5% {hi:.2f}, max {v.max():.2f}; "
              f"within 4.5-27.5: {inside}; spans 5.5-22.5: {spans}; percentiles 1/99: {np.percentile(v, [1, 99]).round(2).tolist()}")

**Result: not reproduced.** At 7 C the central 95% of ratios runs from 7.0 to 26.0 (7 to 11 C: 7.0
to 26.0), inside 4.5 to 27.5 but not reaching down to 5.5; the lowest ratio at 7 C is 5.8 and the
highest 39.9. The upper end matches the paper's 25; the lower end of the bulk is 7, not 5.

## fig6b-lg-phase-locked

> LG burst starts are phase-locked to the PD cycle, their distribution in PD phase sharply peaked
> and significantly non-uniform (Rayleigh test, p < 4e-4, 7.5 < z < 45).

PD phase of each LG burst start (`gastric.measurePhase`), temperature at the PD burst start rounded
to a whole degree, the bins 7 to 23 C. Sample (a), the Methods: the first 40 starts of each
preparation and temperature, combinations with fewer excluded; (b) the later code, first 50;
(c) the archived code, all starts of a temperature pooled, temperatures with fewer than 100
excluded. The Rayleigh test is `circ_rtest`.

In [ ]:
def phase_tables(Bx, slow):
    rows, pooled = [], []
    for j, (p, b) in enumerate(zip(preps, Bx)):
        ph, T = stg.measure_phase(p, b, b[slow]["starts"])
        Tr = np.round(T)
        for c in stg.BIN_CENTRES:
            v = ph[(Tr == c) & np.isfinite(ph)]
            r = {"prep": p["name"], "T": int(c), "n": len(v), "cmean_all": stg.circ_mean01(v)}
            for N in (40, 50):
                if len(v) >= N:
                    r[f"p{N}"], r[f"z{N}"] = stg.rayleigh(v[:N])
                    r[f"cmean{N}"] = stg.circ_mean01(v[:N])
            rows.append(r)
        pooled.append(pd.DataFrame({"prep": j, "T": Tr, "phase": ph}))
    P = pd.concat(pooled)
    P = P[np.isfinite(P.phase)]
    pool = []
    for c in range(5, 26, 2):
        v = P.phase[P["T"] == c].values
        if len(v) >= 100:
            pv, z = stg.rayleigh(v)
            pool.append({"T": c, "n": len(v), "p": pv, "z": z})
    return pd.DataFrame(rows), pd.DataFrame(pool)


def summarise(tab, pool, label):
    out = {}
    for N in (40, 50):
        s = tab.dropna(subset=[f"p{N}"])
        out[N] = s
        print(f"  {label} first {N}: {len(s)} combinations; p {s[f'p{N}'].min():.2g} to {s[f'p{N}'].max():.2g}; "
              f"z {s[f'z{N}'].min():.2f} to {s[f'z{N}'].max():.2f} (median {s[f'z{N}'].median():.1f}); "
              f"p >= 0.05 in {int((s[f'p{N}'] >= 0.05).sum())}")
    print(f"  {label} pooled (>= 100 per temperature):", pool.round(3).to_dict("records"))
    return out

ph_lg = {lab: phase_tables(Bx, "LG") for lab, Bx in (("Methods rule", B), ("code rule", BC))}
for lab, (tab, pool) in ph_lg.items():
    print(f"--- LG, {lab}")
    s = summarise(tab, pool, lab)
    a = s[40]
    print(f"  sample (a): all p < 4.4e-4: {bool((a.p40 < 4.4e-4).all())}; all z in 6.75-49.5: {bool(a.z40.between(6.75, 49.5).all())}; "
          f"all p < 0.05: {bool((a.p40 < 0.05).all())}; combinations failing p < 4.4e-4: {int((a.p40 >= 4.4e-4).sum())}")
    b50 = s[50]
    print(f"  sample (b): all p < 4.4e-4: {bool((b50.p50 < 4.4e-4).all())}; all z in 6.75-49.5: {bool(b50.z50.between(6.75, 49.5).all())}")
tab, _ = ph_lg["Methods rule"]
print("\nSample (a), Methods rule, per combination:")
print(tab.dropna(subset=["p40"])[["prep", "T", "n", "p40", "z40"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: partially reproduced.** Under the Methods' sample (first 40 starts), every one of the 41
preparation and temperature combinations is significantly non-uniform (p < 0.05; largest p 0.008),
but not every one reaches p < 4.4e-4, and z runs from 4.7 to 35.8, below the reported 7.5 at the
low end (Methods burst rule). With the code's burst rule one combination has p = 0.08. The later
code's sample (first 50) with the code's burst rule gives p up to 4.5e-4 and z 7.5 to 45.4, which
are the reported bounds within rounding; with the Methods burst rule and the first 50, p up to
2.1e-4 and z 8.2 to 45.3, within them. The archived code's pooled sample gives z of 67 to 577, far
above them.

## fig6b-lg-start-counts

> A total of 3816 LG burst starts and 3228 DG burst starts across nine temperature points and 10
> preparations enter the phase distributions, and about 35% of preparation and temperature
> combinations have fewer than 40 burst starts.

All LG and DG burst starts with a defined PD phase in the bins 7 to 23 C. The fraction, reading (a):
of preparation and temperature combinations; the claims file does not say over which combinations,
so both those with at least one start and all 90 (10 preparations by 9 temperatures) are shown;
reading (b): of burst starts.

In [ ]:
ph_dg = {lab: phase_tables(Bx, "DG") for lab, Bx in (("Methods rule", B), ("code rule", BC))}
for lab in ("Methods rule", "code rule"):
    lt, lp = ph_lg[lab]
    dt, dp = ph_dg[lab]
    both = pd.concat([lt.assign(neuron="LG"), dt.assign(neuron="DG")])
    nz = both[both.n > 0]
    print(f"--- {lab}: LG starts {lt.n.sum()} (expected 3816 +- 381.6, within: {abs(lt.n.sum() - 3816) <= 381.6}); "
          f"DG starts {dt.n.sum()} (expected 3228 +- 322.8, within: {abs(dt.n.sum() - 3228) <= 322.8})")
    print(f"  pooled over temperatures with >= 100 (code's figure): LG {lp.n.sum()}, DG {dp.n.sum()}")
    for name, t in (("LG", lt), ("DG", dt), ("LG and DG", both)):
        z = t[t.n > 0]
        print(f"  {name}: combinations < 40 of those with any start: {(z.n < 40).mean():.3f} ({int((z.n < 40).sum())}/{len(z)}); "
              f"of all {len(t)}: {(t.n < 40).mean():.3f}; fraction of starts in combinations < 40: {t.n[t.n < 40].sum() / t.n.sum():.3f}")

**Result: not reproduced.** Under the Methods burst rule there are 3908 LG burst starts (within
3816 plus or minus 381.6) and 3573 DG starts, above the tolerance of 3228 plus or minus 322.8 by 22.
The fraction of combinations with fewer than 40 starts is 0.34 for LG and 0.39 for DG over
combinations with any start (0.37 for both together), near the reported ~35%, and 0.09 to 0.10 of the
starts. With the code's burst rule the counts are 3865 LG and 3448 DG, both within tolerance, so
the claim holds under the code's rule but not under the Methods' one, which the claims file sets.
With the code's burst rule and the archived code's sample (temperatures with at least 100 pooled
starts), the totals are 3816 LG, the paper's number exactly, and 3220 DG against 3228.

## fig6b-lg-phase-constant

> From 7 to 21 C, LG bursts start at a phase of about 0.5 in the PD cycle, and this phase does not
> trend with temperature (Spearman test, p = 0.46, rho = -0.31).

Circular mean phase per preparation and bin, then circular mean over preparations
(`groupAndPlotErrorBars` with circular means); Spearman correlation of the eight bin means 7 to
21 C with temperature (absence, reproduced when p >= 0.05); the phase holds when every bin mean is
within 0.45 to 0.55. All starts and the first 40 per combination are both run.

In [ ]:
for lab in ("Methods rule", "code rule"):
    tab, _ = ph_lg[lab]
    for col, name in (("cmean_all", "all starts"), ("cmean40", "first 40")):
        M = np.array([stg.circ_mean01(tab[tab["T"] == c][col].dropna().values) for c in stg.BIN_CENTRES])
        sel = (stg.BIN_CENTRES >= 7) & (stg.BIN_CENTRES <= 21) & np.isfinite(M)
        rho, pval = stats.spearmanr(stg.BIN_CENTRES[sel], M[sel])
        print(f"{lab}, {name}: bin means {dict(zip(stg.BIN_CENTRES.tolist(), M.round(3).tolist()))}")
        print(f"   7-21 C: Spearman rho = {rho:.3f}, p = {pval:.3f}; range {M[sel].min():.3f}-{M[sel].max():.3f}; "
              f"all within 0.45-0.55: {bool(((M[sel] >= 0.45) & (M[sel] <= 0.55)).all())}; bins outside: {stg.BIN_CENTRES[sel][(M[sel] < 0.45) | (M[sel] > 0.55)].tolist()}")

**Result: partially reproduced.** The phase does not trend with temperature from 7 to 21 C (all
starts: rho = -0.19, p = 0.65; first 40: rho = -0.43, p = 0.29; Methods burst rule). With the code's
burst rule and all starts, rho = -0.310 and p = 0.456, the paper's values. The mean phase is near
but mostly below 0.5: 0.42 to 0.49 with all starts, 0.34 to 0.50 with the first 40, so not every bin
lies within 0.45 to 0.55 under either sample.

## fig6c-lg-spiking-drops-near-pd-end

> LG spike probability in the PD cycle shows a pronounced decrease at higher temperatures,
> occurring close to PD burst termination.

PD phase of every LG spike, rounded temperature, temperatures with at least 1000 spikes, 30-bin
p.d.f. on 0 to 1; mean PD burst-end phase per temperature. The value is the p.d.f. in the bin
holding the mean PD end (a), or the lowest bin within 0.1 phase of it (b).

In [ ]:
def spike_pdf_table(Bx, neuron):
    stops, sT, ph, T = [], [], [], []
    for p, b in zip(preps, Bx):
        s, t = stg.measure_phase(p, b, b["PD"]["ends"])
        stops.append(s); sT.append(t)
        a, t2 = stg.measure_phase(p, b, p[neuron])
        ph.append(a); T.append(t2)
    stops, sT = np.concatenate(stops), np.round(np.concatenate(sT))
    ph, T = np.concatenate(ph), np.round(np.concatenate(T))
    edges = np.linspace(0, 1, 31)
    centres = (edges[:-1] + edges[1:]) / 2
    rows = []
    for c in range(5, 26, 2):
        u = (T == c) & np.isfinite(ph)
        if u.sum() < 1000:
            continue
        h, _ = np.histogram(ph[u], edges, density=True)
        M = np.nanmean(stops[sT == c])
        k = min(int(M * 30), 29)
        near = np.abs(centres - M) <= 0.1
        rows.append({"T": c, "spikes": int(u.sum()), "PD_end_phase": M, "pdf_a": h[k], "pdf_b": h[near].min(),
                     "pdf_min": h.min(), "pdf_max": h.max()})
    return pd.DataFrame(rows)

pdf_lg = {lab: spike_pdf_table(Bx, "LG") for lab, Bx in (("Methods rule", B), ("code rule", BC))}
for lab, t in pdf_lg.items():
    print(f"--- LG, {lab}")
    print(t.round(3).to_string(index=False))
    for rd in ("a", "b"):
        rho, pval = stats.spearmanr(t["T"], t[f"pdf_{rd}"])
        print(f"  reading ({rd}): value at warmest ({t['T'].iloc[-1]} C) {t[f'pdf_{rd}'].iloc[-1]:.3f} < 1: {t[f'pdf_{rd}'].iloc[-1] < 1}; "
              f"Spearman with temperature rho = {rho:.3f}, p = {pval:.3f}")
    print("  without the 23 C bin (one or two preparations):",
          {rd: round(stats.spearmanr(t["T"][t["T"] < 23], t[f"pdf_{rd}"][t["T"] < 23])[0], 3) for rd in ("a", "b")})

**Result: not reproduced.** LG spike density at the PD burst end is below the uniform level at
every temperature (0.45 to 0.81 from 7 to 21 C), and lowest at 23 C (0.23, reading a), but it does
not decrease with temperature: the Spearman correlation with temperature is not negative under
either reading (printed above), as the density rises from 11 to 21 C. The drop at 23 C rests on the
two preparations that reach it.

## fig6e-dg-not-phase-locked

> DG bursts are not phase-locked to PD; DG burst starts are not sharply peaked in PD phase, not
> significantly non-uniform in every preparation and temperature, and the effect is much smaller
> than for LG (p from 0.001 to 0.85, 0.15 < z < 13).

The same test with DG burst starts; reproduced when at least one combination has p >= 0.05 and the
median DG z is below the median LG z under the same sample.

In [ ]:
for lab in ("Methods rule", "code rule"):
    dt, dp = ph_dg[lab]
    lt, _ = ph_lg[lab]
    print(f"--- DG, {lab}")
    s = summarise(dt, dp, lab)
    for N in (40, 50):
        d = s[N]
        l = lt.dropna(subset=[f"z{N}"])
        print(f"  first {N}: p >= 0.05 in {int((d[f'p{N}'] >= 0.05).sum())} of {len(d)}; median z DG {d[f'z{N}'].median():.2f} "
              f"against LG {l[f'z{N}'].median():.2f}; DG z within 0.15-13: {bool(d[f'z{N}'].between(0.15, 13).all())}; "
              f"DG p within 0.001-0.85: {bool(d[f'p{N}'].between(0.001, 0.85).all())}")

**Result: reproduced.** Under the Methods' sample (first 40), 16 of 35 DG combinations are not
significantly non-uniform (p >= 0.05), and the median DG z, 3.4, is far below the median LG z, 22.6
(Methods burst rule). DG p ranges from 1.4e-8 to 0.82 and z from 0.21 to 16.25, beyond the reported
0.001 to 0.85 and 0.15 to 13 at the significant end; with the later code's sample of 50, z runs
0.16 to 12.3 (Methods burst rule) or 13.7 (code's), and p from 2e-6 to 0.85.

## fig6f-dg-spiking-no-drop

> DG spike probability does not show a noticeable decrease close to the end of PD bursting.

The measure of fig6c applied to DG spikes; reproduced when, under the reading used for LG (a),
the DG value is not below 1 by more than half of the LG drop at the same temperature, at every
included temperature.

In [ ]:
for lab, Bx in (("Methods rule", B), ("code rule", BC)):
    d = spike_pdf_table(Bx, "DG")
    l = pdf_lg[lab].set_index("T")
    d["LG_pdf_a"] = d["T"].map(l["pdf_a"])
    d["threshold"] = 1 - 0.5 * (1 - d["LG_pdf_a"])
    d["holds"] = d["pdf_a"] >= d["threshold"]
    print(f"--- DG, {lab}")
    print(d.round(3).to_string(index=False))
    print(f"  holds at {int(d.holds.sum())} of {len(d)} temperatures; DG value range {d.pdf_a.min():.2f}-{d.pdf_a.max():.2f}")

**Result: reproduced.** DG spike density at the PD burst-end bin is 0.89 to 1.09 at the eight
temperatures with at least 1000 DG spikes (7 to 21 C), and at each it is above the threshold of 1
minus half of the LG drop (printed above).

## Summary

| Claim | Outcome |
|---|---|
| fig2-spontaneous-11-to-30 | Blocked: data missing |
| fig2a-less-regular-at-high-temperature | Blocked: data missing |
| fig3-evoked-7-to-21-in-9-of-10 | Not reproduced (8 of 10) |
| fig3s1-spontaneous-and-evoked-similar | Blocked: data missing |
| fig4b-periods-decrease-all-preps | Reproduced |
| fig4c-q10-about-2 | Not reproduced (Methods' Q10; the code's Q10 gives 8 of 10) |
| fig4c-q10-pd-lg-not-different | Reproduced |
| fig4s1-slowdown-after-stimulation | Reproduced |
| fig4s2-gastric-without-pyloric-pacemaker | Blocked: data missing |
| fig4s2-rhythm-counts | Blocked: data missing |
| fig5a-example-ratios | Reproduced |
| fig5d-lg-pd-integer-coupled | Reproduced |
| fig5d-dg-pd-not-coupled | Reproduced |
| fig5e-lg-area-exceeds-dg | Partially reproduced |
| fig5f-ratio-independent-of-temperature | Reproduced |
| fig5f-ratio-range | Not reproduced |
| fig6b-lg-phase-locked | Partially reproduced |
| fig6b-lg-start-counts | Not reproduced (DG count 22 over tolerance; holds under the code's burst rule) |
| fig6b-lg-phase-constant | Partially reproduced |
| fig6c-lg-spiking-drops-near-pd-end | Not reproduced |
| fig6e-dg-not-phase-locked | Reproduced |
| fig6f-dg-spiking-no-drop | Reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** Bursts follow the Methods (at least 5 or 2 spikes);
the paper's code keeps bursts with at least that many ISIs, one spike more, and is shown as a
variant throughout. With the code's rule and the code's computations, several reported numbers are
matched exactly (paired t-test t = 1.27, p = 0.235; n = 3799 cycles and Spearman p = 0.88; phase
Spearman rho = -0.31, p = 0.46), so the record holds the data the paper analysed. The Methods'
phase sample (first 40 starts) does not give the reported Rayleigh ranges; the later code's
(first 50) does, and the archived code's pooled sample with the code's burst rule gives the
reported 3816 LG burst starts. The Methods' Q10 does not give the reported count of 8 of 10 near 2; the code's
per-burst Q10 does. Random draws (KS shuffles, bootstraps) use seeds from the claim ids, not the
paper's.

**What was found about the dataset.** `evoked_data.mat` holds the spike times, temperature and
stimulation mask of the ten evoked preparations, consistent with the paper. The spontaneous
recordings and the recordings with the pyloric pacemaker hyperpolarized, behind Figure 2 and
Figure 4, figure supplement 2, are not in the record, although the data statement points only to
it. There is no raw data for 901_005. 932_151, one of the ten evoked preparations, was recorded
in May 2019 by its ABF header, not in 2017 as the Methods say. Two zips (941_003, 941_006, recorded June 2019) belong to no
set the paper describes and have no spike annotations; 941_006 is one of the four preparations of
Figure 3, figure supplement 1, whose spike times are therefore not available. The record has no
README, and the fields of `evoked_data.mat` (the mask in particular) are not described. Its
`experiment_idx` and `filename` fields are MATLAB categorical objects, readable outside MATLAB only by
decoding the MCOS subsystem. The fields `T` and `time_offset` are empty.